In [4]:
!pip install -q earthengine-api geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 13.6 MB/s eta 0:00:00


In [1]:
import ee
import geemap

print("Earth Engine library loaded!")

Earth Engine library loaded!


In [3]:
ee.Authenticate()

In [5]:
ee.Initialize(project="urban-heat-ai-ml")

print("Earth Engine connected!")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Earth Engine connected!


In [6]:
print(ee.Number(10).add(20).getInfo())

30


In [7]:
# Load Delhi NCT boundary
delhi = ee.FeatureCollection("FAO/GAUL/2015/level1") \
    .filter(ee.Filter.eq("ADM1_NAME", "Delhi"))

print("Delhi boundary loaded!")
print("Number of features:", delhi.size().getInfo())

Delhi boundary loaded!


/usr/local/lib/python3.13/dist-packages/ee/deprecation.py:215: DeprecationWarning: 

Attention required for FAO/GAUL/2015/level1! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by FAO/GAUL/2025/level1

Learn more: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_2015_level1

  warnings.warn(warning, category=DeprecationWarning)


Number of features: 1


In [9]:
Map = geemap.Map()

Map.centerObject(delhi, 10)
Map.addLayer(delhi, {}, "Delhi NCT")

Map

Map(center=[28.646455931994893, 77.10900424302301], controls=(WidgetControl(options=['position', 'transparent_…

In [10]:
print("Delhi features:", delhi.size().getInfo())
print("Delhi geometry:", delhi.geometry().area().getInfo())

Delhi features: 1
Delhi geometry: 1504310666.6713402


In [11]:
landsat = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .filterBounds(delhi)
    .filterDate("2024-05-01", "2024-06-01")
)

print("Landsat images in May 2024:", landsat.size().getInfo())

Landsat images in May 2024: 4


In [12]:
image = ee.Image(landsat.first())

print("Image ID:", image.id().getInfo())
print("Bands:", image.bandNames().getInfo())

Image ID: LC08_146040_20240511
Bands: ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'SR_QA_AEROSOL', 'ST_B10', 'ST_ATRAN', 'ST_CDIST', 'ST_DRAD', 'ST_EMIS', 'ST_EMSD', 'ST_QA', 'ST_TRAD', 'ST_URAD', 'QA_PIXEL', 'QA_RADSAT']


In [13]:
# Convert Landsat surface temperature to Celsius
lst = image.select("ST_B10").multiply(0.00341802).add(149.0).subtract(273.15)

print("LST band created successfully!")

LST band created successfully!


In [14]:
stats = lst.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=delhi.geometry(),
    scale=30,
    maxPixels=1e9
)

print(stats.getInfo())

{'ST_B10_max': 59.835180560000026, 'ST_B10_min': -34.43722905999999}


In [15]:
# Create a cloud/shadow mask from QA_PIXEL
qa = image.select("QA_PIXEL")

cloud_shadow_mask = (
    qa.bitwiseAnd(1 << 3).eq(0)   # cloud
    .And(qa.bitwiseAnd(1 << 4).eq(0))  # cloud shadow
    .And(qa.bitwiseAnd(1 << 5).eq(0))  # snow
)

# Apply the mask to LST
lst_masked = lst.updateMask(cloud_shadow_mask)

print("Cloud/shadow mask applied!")

Cloud/shadow mask applied!


In [16]:
stats_masked = lst_masked.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=delhi.geometry(),
    scale=30,
    maxPixels=1e9
)

print(stats_masked.getInfo())

{'ST_B10_max': 59.835180560000026, 'ST_B10_min': -11.269889499999977}


In [17]:
# Rebuild the cloud/shadow mask properly
qa = image.select("QA_PIXEL")

clear_mask = (
    qa.bitwiseAnd(1 << 0).eq(0)   # Fill
    .And(qa.bitwiseAnd(1 << 1).eq(0))  # Dilated cloud
    .And(qa.bitwiseAnd(1 << 2).eq(0))  # Cirrus
    .And(qa.bitwiseAnd(1 << 3).eq(0))  # Cloud
    .And(qa.bitwiseAnd(1 << 4).eq(0))  # Cloud shadow
    .And(qa.bitwiseAnd(1 << 5).eq(0))  # Snow
)

# ST uncertainty: 0.01 K per digital unit
st_qa = image.select("ST_QA")
quality_mask = st_qa.multiply(0.01).lte(5)

# Final mask
final_mask = clear_mask.And(quality_mask)

lst_clean = lst.updateMask(final_mask)

print("Final LST mask applied!")

Final LST mask applied!


In [18]:
stats_clean = lst_clean.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=delhi.geometry(),
    scale=30,
    maxPixels=1e9
)

print(stats_clean.getInfo())

{'ST_B10_max': 59.835180560000026, 'ST_B10_min': 31.472450600000002}


In [19]:
Map = geemap.Map()

Map.centerObject(delhi, 10)

Map.addLayer(
    lst_clean,
    {
        "min": 30,
        "max": 60,
        "palette": ["blue", "cyan", "green", "yellow", "orange", "red"]
    },
    "Clean LST May 2024"
)

Map.addLayer(delhi, {}, "Delhi NCT")

Map

Map(center=[28.646455931994893, 77.10900424302301], controls=(WidgetControl(options=['position', 'transparent_…

In [20]:
Map = geemap.Map(add_google_map=False)

Map.centerObject(delhi, 10)

Map.addLayer(
    lst_clean,
    {
        "min": 30,
        "max": 60,
        "palette": ["blue", "cyan", "green", "yellow", "orange", "red"]
    },
    "Clean LST May 2024"
)

Map.addLayer(delhi, {}, "Delhi NCT")

Map

Map(center=[28.646455931994893, 77.10900424302301], controls=(WidgetControl(options=['position', 'transparent_…

In [21]:
# Load Sentinel-2 Surface Reflectance imagery for May 2024
sentinel = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(delhi)
    .filterDate("2024-05-01", "2024-06-01")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30))
)

print("Sentinel-2 images in May 2024:", sentinel.size().getInfo())

Sentinel-2 images in May 2024: 20


In [22]:
def mask_s2(image):
    scl = image.select("SCL")

    # Keep:
    # 4 = Vegetation
    # 5 = Bare soil
    # 6 = Water
    # 7 = Unclassified
    mask = (
        scl.eq(4)
        .Or(scl.eq(5))
        .Or(scl.eq(6))
        .Or(scl.eq(7))
    )

    return image.updateMask(mask)


sentinel_clean = sentinel.map(mask_s2)

sentinel_may = sentinel_clean.median().clip(delhi)

print("May 2024 Sentinel-2 composite created!")

May 2024 Sentinel-2 composite created!


In [23]:
ndvi = sentinel_may.normalizedDifference(["B8", "B4"]).rename("NDVI")

print("NDVI calculated successfully!")

NDVI calculated successfully!


In [24]:
ndvi_stats = ndvi.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=delhi.geometry(),
    scale=10,
    maxPixels=1e9
)

print(ndvi_stats.getInfo())

{'NDVI_max': 0.8718975180144115, 'NDVI_min': -0.3223872817305186}


In [25]:
# Calculate NDBI
ndbi = sentinel_may.normalizedDifference(["B11", "B8"]).rename("NDBI")

# Calculate NDWI
ndwi = sentinel_may.normalizedDifference(["B3", "B8"]).rename("NDWI")

# Sentinel-2 reflectance
nir = sentinel_may.select("B8").divide(10000)
red = sentinel_may.select("B4").divide(10000)
blue = sentinel_may.select("B2").divide(10000)

# EVI denominator
denominator = (
    nir
    .add(red.multiply(6))
    .subtract(blue.multiply(7.5))
    .add(1)
)

# EVI formula
evi_raw = (
    nir.subtract(red)
    .multiply(2.5)
    .divide(denominator)
)

# Keep only stable, physically reasonable EVI pixels
evi = (
    evi_raw
    .updateMask(denominator.gt(0.1))
    .updateMask(evi_raw.gte(-1))
    .updateMask(evi_raw.lte(1))
    .rename("EVI")
)

print("EVI calculated and quality-masked!")

print("NDBI, NDWI and EVI calculated successfully!")

EVI calculated and quality-masked!
NDBI, NDWI and EVI calculated successfully!


In [26]:
spectral_stats = ee.Image.cat([ndbi, ndwi, evi]).reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=delhi.geometry(),
    scale=20,
    maxPixels=1e9
)

print(spectral_stats.getInfo())


{'EVI_max': 0.9998710259882629, 'EVI_min': -0.6292219164373265, 'NDBI_max': 0.47613997879109227, 'NDBI_min': -0.5531686358754028, 'NDWI_max': 0.4340712223291627, 'NDWI_min': -0.7966234786022772}


In [27]:
# Combine the four spectral features into one image
spectral_features = ee.Image.cat([
    ndvi,
    ndbi,
    ndwi,
    evi
])

print("Spectral feature image created!")
print("Bands:", spectral_features.bandNames().getInfo())

Spectral feature image created!
Bands: ['NDVI', 'NDBI', 'NDWI', 'EVI']


In [28]:
# Load SRTM Digital Elevation Model
dem = ee.Image("USGS/SRTMGL1_003").clip(delhi)

print("SRTM DEM loaded!")

SRTM DEM loaded!


In [29]:
# Calculate terrain features
terrain = ee.Terrain.products(dem)

elevation = terrain.select("elevation").rename("elevation_m")
slope = terrain.select("slope").rename("slope_deg")
aspect = terrain.select("aspect").rename("aspect_deg")

print("Elevation, slope and aspect calculated!")
print("Bands:", ee.Image.cat([
    elevation, slope, aspect
]).bandNames().getInfo())

Elevation, slope and aspect calculated!
Bands: ['elevation_m', 'slope_deg', 'aspect_deg']


In [30]:
# Global Surface Water occurrence
water_occurrence = (
    ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
    .select("occurrence")
)

# Water pixels: occurrence >= 50%
water = (
    water_occurrence
    .gte(50)
    .selfMask()
    .clip(delhi)
)

# Prepare at 30 m scale
water_30m = water.reproject(
    crs="EPSG:4326",
    scale=30
)

# Distance to nearest water pixel
dist_water = (
    water_30m
    .mask()
    .fastDistanceTransform(1024, "pixels", "squared_euclidean")
    .sqrt()
    .multiply(30)
    .rename("dist_water_m")
    .clip(delhi)
)

print("Distance-to-water layer rebuilt!")

Distance-to-water layer rebuilt!


In [31]:
water_stats = dist_water.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=delhi.geometry(),
    scale=30,
    maxPixels=1e9
)

print(water_stats.getInfo())

{'dist_water_m_max': 22590.697200396447, 'dist_water_m_min': 0}


In [32]:
# Load ESA WorldCover 2021
worldcover = (
    ee.ImageCollection("ESA/WorldCover/v200")
    .first()
    .select("Map")
    .clip(delhi)
)

print("ESA WorldCover loaded!")
print("Band:", worldcover.bandNames().getInfo())


ESA WorldCover loaded!
Band: ['Map']


In [33]:
# Check land-cover class values
landcover_stats = worldcover.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=delhi.geometry(),
    scale=10,
    maxPixels=1e9
)

print(landcover_stats.getInfo())

{'Map': {'10': 3200374.3450980396, '20': 520845.06666666665, '30': 655615.4509803922, '40': 5364534.909803922, '50': 7072403.054901961, '60': 234418.0862745098, '80': 130833.30588235294, '90': 673}}


In [34]:
# Load GFCC Global 30m Tree Cover
gfcc = (
    ee.ImageCollection("NASA/MEASURES/GFCC/TC/v3")
    .filterBounds(delhi)
    .filter(ee.Filter.eq("year", 2015))
)

print("GFCC images:", gfcc.size().getInfo())

# Get the 2015 tree-cover image
gfcc_2015 = ee.Image(gfcc.first())

print("GFCC bands:", gfcc_2015.bandNames().getInfo())

GFCC images: 2
GFCC bands: ['tree_canopy_cover', 'uncertainty']


In [35]:
# Create tree cover percentage layer
tree_cover = (
    gfcc_2015
    .select("tree_canopy_cover")
    .rename("tree_cover_pct")
    .clip(delhi)
)

print("Tree cover percentage layer created!")
print("Band:", tree_cover.bandNames().getInfo())

Tree cover percentage layer created!
Band: ['tree_cover_pct']


In [36]:
tree_stats = tree_cover.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=delhi.geometry(),
    scale=30,
    maxPixels=1e9
)

print(tree_stats.getInfo())

{'tree_cover_pct_max': 20, 'tree_cover_pct_min': 0}


In [37]:
# Load GHSL Global Population Surface
population = (
    ee.Image("JRC/GHSL/P2023A/GHS_POP/2020")
    .clip(delhi)
)

print("GHSL population layer loaded!")
print("Bands:", population.bandNames().getInfo())

GHSL population layer loaded!
Bands: ['population_count']


In [38]:
population_stats = population.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=delhi.geometry(),
    scale=100,
    maxPixels=1e9
)

print(population_stats.getInfo())

{'population_count_max': 2473.87255859375, 'population_count_min': 0}


In [39]:
# Check the native projection and pixel scale
print("Projection:", population.projection().getInfo())
print("Nominal scale (meters):", population.projection().nominalScale().getInfo())

Projection: {'type': 'Projection', 'wkt': 'PROJCS["World_Mollweide", \n  GEOGCS["WGS 84", \n    DATUM["WGS_1984", \n      SPHEROID["WGS 84", 6378137.0, 298.257223563, AUTHORITY["EPSG","7030"]], \n      AUTHORITY["EPSG","6326"]], \n    PRIMEM["Greenwich", 0.0], \n    UNIT["degree", 0.017453292519943295], \n    AXIS["Longitude", EAST], \n    AXIS["Latitude", NORTH]], \n  PROJECTION["Mollweide"], \n  PARAMETER["semi_minor", 6378137.0], \n  PARAMETER["false_easting", 0.0], \n  PARAMETER["false_northing", 0.0], \n  PARAMETER["central_meridian", 0.0], \n  UNIT["m", 1.0], \n  AXIS["Easting", EAST], \n  AXIS["Northing", NORTH]]', 'transform': [100, 0, -18041000, 0, -100, 9000000]}
Nominal scale (meters): 100


In [40]:
# Convert population count to population density (people per km²)
population_density = (
    population
    .divide(0.01)
    .rename("population_density")
    .clip(delhi)
)

print("Population density layer created!")
print("Band:", population_density.bandNames().getInfo())

Population density layer created!
Band: ['population_density']


In [41]:
pop_density_stats = population_density.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=delhi.geometry(),
    scale=100,
    maxPixels=1e9
)

print(pop_density_stats.getInfo())

{'population_density_max': 247387.255859375, 'population_density_min': 0}


In [42]:
# Load ERA5-Land hourly data for May 2024
era5 = (
    ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")
    .filterBounds(delhi)
    .filterDate("2024-05-01", "2024-06-01")
)

print("ERA5-Land images in May 2024:", era5.size().getInfo())

ERA5-Land images in May 2024: 744


In [43]:
era5_first = ee.Image(era5.first())

print("ERA5 bands:")
print(era5_first.bandNames().getInfo())

ERA5 bands:
['dewpoint_temperature_2m', 'temperature_2m', 'skin_temperature', 'soil_temperature_level_1', 'soil_temperature_level_2', 'soil_temperature_level_3', 'soil_temperature_level_4', 'lake_bottom_temperature', 'lake_ice_depth', 'lake_ice_temperature', 'lake_mix_layer_depth', 'lake_mix_layer_temperature', 'lake_shape_factor', 'lake_total_layer_temperature', 'snow_albedo', 'snow_cover', 'snow_density', 'snow_depth', 'snow_depth_water_equivalent', 'snowfall', 'snowmelt', 'temperature_of_snow_layer', 'skin_reservoir_content', 'volumetric_soil_water_layer_1', 'volumetric_soil_water_layer_2', 'volumetric_soil_water_layer_3', 'volumetric_soil_water_layer_4', 'forecast_albedo', 'surface_latent_heat_flux', 'surface_net_solar_radiation', 'surface_net_thermal_radiation', 'surface_sensible_heat_flux', 'surface_solar_radiation_downwards', 'surface_thermal_radiation_downwards', 'evaporation_from_bare_soil', 'evaporation_from_open_water_surfaces_excluding_oceans', 'evaporation_from_the_top_of_

In [44]:
# May 2024 mean air temperature
air_temp_era5 = (
    era5
    .select("temperature_2m")
    .mean()
    .subtract(273.15)
    .rename("air_temp_era5")
    .clip(delhi)
)

print("Air temperature layer created!")

Air temperature layer created!


In [45]:
# Calculate 10 m wind speed for each hourly image
def add_wind_speed(image):
    u = image.select("u_component_of_wind_10m")
    v = image.select("v_component_of_wind_10m")

    wind = u.pow(2).add(v.pow(2)).sqrt().rename("wind_speed")

    return image.addBands(wind)

era5_wind = era5.map(add_wind_speed)

wind_speed = (
    era5_wind
    .select("wind_speed")
    .mean()
    .rename("wind_speed")
    .clip(delhi)
)

print("Wind speed layer created!")

Wind speed layer created!


In [46]:
# Calculate relative humidity from air temperature and dew point
def add_relative_humidity(image):
    temp_c = image.select("temperature_2m").subtract(273.15)
    dew_c = image.select("dewpoint_temperature_2m").subtract(273.15)

    # Magnus formula
    es = temp_c.expression(
        "6.112 * exp((17.67 * T) / (T + 243.5))",
        {"T": temp_c}
    )

    e = dew_c.expression(
        "6.112 * exp((17.67 * Td) / (Td + 243.5))",
        {"Td": dew_c}
    )

    rh = e.divide(es).multiply(100).rename("humidity")

    return image.addBands(rh)

era5_humidity = era5.map(add_relative_humidity)

humidity = (
    era5_humidity
    .select("humidity")
    .mean()
    .rename("humidity")
    .clip(delhi)
)

print("Humidity layer created!")

Humidity layer created!


In [47]:
weather_features = ee.Image.cat([
    air_temp_era5,
    humidity,
    wind_speed
])

weather_stats = weather_features.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=delhi.geometry(),
    scale=11100,
    maxPixels=1e9
)

print(weather_stats.getInfo())

{'air_temp_era5_max': 34.29093386332198, 'air_temp_era5_min': 33.45280172286499, 'humidity_max': 31.092944659509403, 'humidity_min': 26.99449974617061, 'wind_speed_max': 2.910047649517188, 'wind_speed_min': 2.8182547903699726}


In [48]:
# Function to mask clouds/shadows and convert ST_B10 to Celsius
def prepare_landsat_lst(image):
    qa = image.select("QA_PIXEL")

    clear_mask = (
        qa.bitwiseAnd(1 << 0).eq(0)   # Fill
        .And(qa.bitwiseAnd(1 << 1).eq(0))  # Dilated cloud
        .And(qa.bitwiseAnd(1 << 2).eq(0))  # Cirrus
        .And(qa.bitwiseAnd(1 << 3).eq(0))  # Cloud
        .And(qa.bitwiseAnd(1 << 4).eq(0))  # Cloud shadow
        .And(qa.bitwiseAnd(1 << 5).eq(0))  # Snow
    )

    st_qa = image.select("ST_QA")

    quality_mask = st_qa.multiply(0.01).lte(5)

    final_mask = clear_mask.And(quality_mask)

    lst_celsius = (
        image.select("ST_B10")
        .multiply(0.00341802)
        .add(149.0)
        .subtract(273.15)
        .rename("actual_lst")
    )

    return lst_celsius.updateMask(final_mask)


# Prepare all May Landsat scenes
landsat_lst = landsat.map(prepare_landsat_lst)

# Monthly median LST composite
may_lst = landsat_lst.median().clip(delhi)

print("May 2024 LST composite created!")
print("Band:", may_lst.bandNames().getInfo())

May 2024 LST composite created!
Band: ['actual_lst']


In [49]:
may_lst_stats = may_lst.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=delhi.geometry(),
    scale=30,
    maxPixels=1e9
)

print(may_lst_stats.getInfo())

{'actual_lst_max': 59.90012294000002, 'actual_lst_min': 33.280583180000036}


In [50]:
# Combine all project features into one image

feature_image = ee.Image.cat([
    may_lst,              # actual_lst
    ndvi,                 # NDVI
    ndbi,                 # NDBI
    ndwi,                 # NDWI
    evi,                  # EVI
    tree_cover,           # tree_cover_pct
    worldcover.rename("land_cover_class"),
    population_density,   # population_density
    elevation,            # elevation_m
    slope,                # slope_deg
    aspect,               # aspect_deg
    dist_water,           # dist_water_m
    air_temp_era5,        # air_temp_era5
    humidity,             # humidity
    wind_speed            # wind_speed
]).clip(delhi)

print("Complete feature image created!")
print("Bands:")
print(feature_image.bandNames().getInfo())

Complete feature image created!
Bands:
['actual_lst', 'NDVI', 'NDBI', 'NDWI', 'EVI', 'tree_cover_pct', 'land_cover_class', 'population_density', 'elevation_m', 'slope_deg', 'aspect_deg', 'dist_water_m', 'air_temp_era5', 'humidity', 'wind_speed']


In [ ]:
# Sample 5,000 random 30 m observations over Delhi
may_samples_test = feature_image.sample(
    region=delhi.geometry(),
    scale=30,
    numPixels=5000,
    seed=42,
    geometries=True,
    tileScale=4
)

print("Test sample created!")
print("Number of samples:", may_samples_test.size().getInfo())

Test sample created!


KeyboardInterrupt: 

In [ ]:
# Inspect one sample
print(may_samples_test.first().getInfo())

In [ ]:
import pandas as pd

sample_list = may_samples_test.getInfo()["features"]

rows = []

for i, feature in enumerate(sample_list):
    props = feature["properties"]
    coords = feature["geometry"]["coordinates"]

    props["cell_id"] = f"DELHI_MAY_2024_{i:05d}"
    props["longitude"] = coords[0]
    props["latitude"] = coords[1]
    props["month_timestamp"] = "2024-05-01"

    rows.append(props)

df_may_test = pd.DataFrame(rows)

# Put metadata columns first
metadata_cols = ["cell_id", "latitude", "longitude", "month_timestamp"]

feature_cols = [
    "NDVI",
    "NDBI",
    "NDWI",
    "EVI",
    "tree_cover_pct",
    "land_cover_class",
    "population_density",
    "elevation_m",
    "slope_deg",
    "aspect_deg",
    "dist_water_m",
    "air_temp_era5",
    "humidity",
    "wind_speed",
    "actual_lst"
]

df_may_test = df_may_test[metadata_cols + feature_cols]

print("Shape:", df_may_test.shape)
print("\nColumns:")
print(df_may_test.columns.tolist())

display(df_may_test.head())

In [ ]:
# Create the full May 2024 sample
may_samples = feature_image.sample(
    region=delhi.geometry(),
    scale=30,
    numPixels=55000,
    seed=2024,
    geometries=True,
    tileScale=4
)

print("Requested samples: 55000")
print("Valid samples returned:", may_samples.size().getInfo())

In [ ]:
def add_metadata(feature):
    coords = feature.geometry().coordinates()

    longitude = ee.Number(coords.get(0))
    latitude = ee.Number(coords.get(1))

    cell_id = (
        ee.String("DELHI_MAY_2024_")
        .cat(longitude.format("%.6f"))
        .cat("_")
        .cat(latitude.format("%.6f"))
    )

    return feature.set({
        "cell_id": cell_id,
        "latitude": latitude,
        "longitude": longitude,
        "month_timestamp": "2024-05-01"
    })


may_samples_export = may_samples.map(add_metadata)

print(
    "Export-ready samples:",
    may_samples_export.size().getInfo()
)

In [ ]:
print(may_samples_export.first().getInfo())

In [ ]:
# Export FINAL May 2024 dataset to Google Drive

export_task_may_final = ee.batch.Export.table.toDrive(
    collection=may_samples_export,
    description="Delhi_Urban_Heat_May_2024_FINAL",
    folder="Urban_Heat_AI_ML",
    fileNamePrefix="Delhi_Urban_Heat_May_2024_FINAL",
    fileFormat="CSV",
    selectors=[
        "cell_id",
        "latitude",
        "longitude",
        "month_timestamp",
        "NDVI",
        "NDBI",
        "NDWI",
        "EVI",
        "tree_cover_pct",
        "land_cover_class",
        "population_density",
        "elevation_m",
        "slope_deg",
        "aspect_deg",
        "dist_water_m",
        "air_temp_era5",
        "humidity",
        "wind_speed",
        "actual_lst"
    ]
)

export_task_may_final.start()

print("Final May export started!")
print("Task ID:", export_task_may_final.id)

In [ ]:
# Check FINAL May export status

for t in ee.batch.Task.list():
    if t.id == "DELJVAQZTWGRYQKQ54UMICFC":
        print("Task status:", t.status())

In [ ]:
# ==========================================
# JUNE 2024 DATASET
# ==========================================

In [ ]:
# Load Landsat 8 Level-2 imagery for June 2024

landsat_june = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .filterBounds(delhi)
    .filterDate("2024-06-01", "2024-07-01")
)

print(
    "Landsat 8 images in June 2024:",
    landsat_june.size().getInfo()
)

Landsat 8 images in June 2024: 5


In [ ]:
# ==========================================
# JUNE 2024 — LAND SURFACE TEMPERATURE
# ==========================================

# Prepare all June Landsat scenes
landsat_lst_june = landsat_june.map(prepare_landsat_lst)

# Create June 2024 monthly median LST
lst_june = (
    landsat_lst_june
    .median()
    .clip(delhi)
    .rename("actual_lst")
)

print("June 2024 LST composite created!")
print("Band:", lst_june.bandNames().getInfo())

June 2024 LST composite created!
Band: ['actual_lst']


In [ ]:
june_lst_stats = lst_june.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=delhi.geometry(),
    scale=30,
    maxPixels=1e9
)

print(june_lst_stats.getInfo())

{'actual_lst_max': 60.07444196000006, 'actual_lst_min': 34.295735120000046}


In [ ]:
# ==========================================
# JUNE 2024 — SENTINEL-2
# ==========================================

sentinel_june = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(delhi)
    .filterDate("2024-06-01", "2024-07-01")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30))
)

print(
    "Sentinel-2 images in June 2024:",
    sentinel_june.size().getInfo()
)

Sentinel-2 images in June 2024: 19


In [ ]:
# ==========================================
# JUNE 2024 — SENTINEL-2 COMPOSITE
# ==========================================

sentinel_clean_june = sentinel_june.map(mask_s2)

sentinel_median_june = (
    sentinel_clean_june
    .median()
    .clip(delhi)
)

print("June 2024 Sentinel-2 composite created!")
print("Bands:", sentinel_median_june.bandNames().getInfo())

June 2024 Sentinel-2 composite created!
Bands: ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE']


In [ ]:
# ==========================================
# JUNE 2024 — SPECTRAL INDICES
# ==========================================

# NDVI
ndvi_june = (
    sentinel_median_june
    .normalizedDifference(["B8", "B4"])
    .rename("NDVI")
)

# NDBI
ndbi_june = (
    sentinel_median_june
    .normalizedDifference(["B11", "B8"])
    .rename("NDBI")
)

# NDWI
ndwi_june = (
    sentinel_median_june
    .normalizedDifference(["B3", "B8"])
    .rename("NDWI")
)

# Convert Sentinel-2 scaled bands to reflectance
nir_june = sentinel_median_june.select("B8").divide(10000)
red_june = sentinel_median_june.select("B4").divide(10000)
blue_june = sentinel_median_june.select("B2").divide(10000)

# EVI denominator
evi_denominator_june = (
    nir_june
    .add(red_june.multiply(6))
    .subtract(blue_june.multiply(7.5))
    .add(1)
)

# Raw EVI
evi_raw_june = (
    nir_june.subtract(red_june)
    .multiply(2.5)
    .divide(evi_denominator_june)
)

# Keep stable and physically reasonable EVI values
evi_june = (
    evi_raw_june
    .updateMask(evi_denominator_june.gt(0.1))
    .updateMask(evi_raw_june.gte(-1))
    .updateMask(evi_raw_june.lte(1))
    .rename("EVI")
)

print("June NDVI, NDBI, NDWI and EVI calculated!")

June NDVI, NDBI, NDWI and EVI calculated!


In [ ]:
june_spectral_stats = ee.Image.cat([
    ndvi_june,
    ndbi_june,
    ndwi_june,
    evi_june
]).reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=delhi.geometry(),
    scale=20,
    maxPixels=1e9
)

print(june_spectral_stats.getInfo())

{'EVI_max': 0.9997395833333337, 'EVI_min': -0.9029049986914385, 'NDBI_max': 0.5972602739726027, 'NDBI_min': -0.6171941106786258, 'NDVI_max': 1, 'NDVI_min': -0.35068912710566613, 'NDWI_max': 0.4945565394106547, 'NDWI_min': -1}


In [ ]:
# ==========================================
# JUNE 2024 — ERA5-LAND WEATHER
# ==========================================

era5_june = (
    ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")
    .filterBounds(delhi)
    .filterDate("2024-06-01", "2024-07-01")
)

print(
    "ERA5-Land images in June 2024:",
    era5_june.size().getInfo()
)

ERA5-Land images in June 2024: 720


In [ ]:
# ==========================================
# JUNE 2024 — AIR TEMPERATURE
# ==========================================

air_temp_era5_june = (
    era5_june
    .select("temperature_2m")
    .mean()
    .subtract(273.15)
    .rename("air_temp_era5")
    .clip(delhi)
)

print("June air temperature layer created!")

June air temperature layer created!


In [ ]:
# ==========================================
# JUNE 2024 — WIND SPEED
# ==========================================

def add_wind_speed_june(image):
    u = image.select("u_component_of_wind_10m")
    v = image.select("v_component_of_wind_10m")

    wind = (
        u.pow(2)
        .add(v.pow(2))
        .sqrt()
        .rename("wind_speed")
    )

    return image.addBands(wind)


era5_wind_june = era5_june.map(add_wind_speed_june)

wind_speed_june = (
    era5_wind_june
    .select("wind_speed")
    .mean()
    .rename("wind_speed")
    .clip(delhi)
)

print("June wind speed layer created!")

June wind speed layer created!


In [ ]:
# ==========================================
# JUNE 2024 — RELATIVE HUMIDITY
# ==========================================

def add_relative_humidity_june(image):
    temp_c = image.select("temperature_2m").subtract(273.15)
    dew_c = image.select("dewpoint_temperature_2m").subtract(273.15)

    es = temp_c.expression(
        "6.112 * exp((17.67 * T) / (T + 243.5))",
        {"T": temp_c}
    )

    e = dew_c.expression(
        "6.112 * exp((17.67 * Td) / (Td + 243.5))",
        {"Td": dew_c}
    )

    rh = (
        e.divide(es)
        .multiply(100)
        .rename("humidity")
    )

    return image.addBands(rh)


era5_humidity_june = era5_june.map(add_relative_humidity_june)

humidity_june = (
    era5_humidity_june
    .select("humidity")
    .mean()
    .rename("humidity")
    .clip(delhi)
)

print("June humidity layer created!")

June humidity layer created!


In [ ]:
# ==========================================
# JUNE 2024 — WEATHER VALIDATION
# ==========================================

weather_features_june = ee.Image.cat([
    air_temp_era5_june,
    humidity_june,
    wind_speed_june
])

weather_stats_june = weather_features_june.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=delhi.geometry(),
    scale=11100,
    maxPixels=1e9
)

print(weather_stats_june.getInfo())

{'air_temp_era5_max': 35.07886437310117, 'air_temp_era5_min': 34.477890523274766, 'humidity_max': 42.58160690697325, 'humidity_min': 40.07986463719769, 'wind_speed_max': 3.181258497840221, 'wind_speed_min': 3.085728366273112}


In [ ]:
# ==========================================
# JUNE 2024 — COMPLETE FEATURE IMAGE
# ==========================================

feature_image_june = ee.Image.cat([
    lst_june,                    # actual_lst
    ndvi_june,                   # NDVI
    ndbi_june,                   # NDBI
    ndwi_june,                   # NDWI
    evi_june,                    # EVI
    tree_cover,                  # tree_cover_pct
    worldcover.rename("land_cover_class"),
    population_density,          # population_density
    elevation,                   # elevation_m
    slope,                       # slope_deg
    aspect,                      # aspect_deg
    dist_water,                  # dist_water_m
    air_temp_era5_june,          # air_temp_era5
    humidity_june,               # humidity
    wind_speed_june              # wind_speed
]).clip(delhi)

print("June feature image created!")
print("Bands:", feature_image_june.bandNames().getInfo())

June feature image created!
Bands: ['actual_lst', 'NDVI', 'NDBI', 'NDWI', 'EVI', 'tree_cover_pct', 'land_cover_class', 'population_density', 'elevation_m', 'slope_deg', 'aspect_deg', 'dist_water_m', 'air_temp_era5', 'humidity', 'wind_speed']


In [ ]:
# ==========================================
# JUNE 2024 — TEST SAMPLE
# ==========================================

june_samples_test = feature_image_june.sample(
    region=delhi,
    scale=30,
    numPixels=5000,
    seed=42,
    geometries=True
)

print(
    "June test sample created! Number of samples:",
    june_samples_test.size().getInfo()
)

June test sample created! Number of samples: 4984


In [ ]:
# Inspect one June sample

print(june_samples_test.first().getInfo())

{'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [77.20434884009336, 28.63271997019605]}, 'id': '0', 'properties': {'EVI': 0.23136963389704518, 'NDBI': -0.0747008731270885, 'NDVI': 0.33288770053475936, 'NDWI': -0.3357449088960343, 'actual_lst': 42.81002294000001, 'air_temp_era5': 34.69166547987197, 'aspect_deg': 217, 'dist_water_m': 5220, 'elevation_m': 223, 'humidity': 41.61928659698926, 'land_cover_class': 50, 'population_density': 19028.785705566406, 'slope_deg': 3, 'tree_cover_pct': 7, 'wind_speed': 3.1479424553375353}}


In [ ]:
# Check June test sample for missing values

june_first = june_samples_test.first()

print("Checking June test sample for missing values...\n")

bands_to_check_june = [
    "actual_lst",
    "NDVI",
    "NDBI",
    "NDWI",
    "EVI",
    "tree_cover_pct",
    "land_cover_class",
    "population_density",
    "elevation_m",
    "slope_deg",
    "aspect_deg",
    "dist_water_m",
    "air_temp_era5",
    "humidity",
    "wind_speed"
]

for band in bands_to_check_june:
    value = june_first.get(band).getInfo()
    print(f"{band}: {value}")

Checking June test sample for missing values...

actual_lst: 42.81002294000001
NDVI: 0.33288770053475936
NDBI: -0.0747008731270885
NDWI: -0.3357449088960343
EVI: 0.23136963389704518
tree_cover_pct: 7
land_cover_class: 50
population_density: 19028.785705566406
elevation_m: 223
slope_deg: 3
aspect_deg: 217
dist_water_m: 5220
air_temp_era5: 34.69166547987197
humidity: 41.61928659698926
wind_speed: 3.1479424553375353


In [ ]:
# Check all June test samples for missing values

june_missing = june_samples_test.filter(
    ee.Filter.Or(
        ee.Filter.notNull(["actual_lst"]).Not(),
        ee.Filter.notNull(["NDVI"]).Not(),
        ee.Filter.notNull(["NDBI"]).Not(),
        ee.Filter.notNull(["NDWI"]).Not(),
        ee.Filter.notNull(["EVI"]).Not(),
        ee.Filter.notNull(["tree_cover_pct"]).Not(),
        ee.Filter.notNull(["land_cover_class"]).Not(),
        ee.Filter.notNull(["population_density"]).Not(),
        ee.Filter.notNull(["elevation_m"]).Not(),
        ee.Filter.notNull(["slope_deg"]).Not(),
        ee.Filter.notNull(["aspect_deg"]).Not(),
        ee.Filter.notNull(["dist_water_m"]).Not(),
        ee.Filter.notNull(["air_temp_era5"]).Not(),
        ee.Filter.notNull(["humidity"]).Not(),
        ee.Filter.notNull(["wind_speed"]).Not()
    )
)

print("June samples with at least one missing value:",
      june_missing.size().getInfo())

print("Total June test samples:",
      june_samples_test.size().getInfo())

June samples with at least one missing value: 0
Total June test samples: 4984


In [ ]:
# Calculate statistics for all June test-sample features

june_stats = june_samples_test.reduceColumns(
    reducer=ee.Reducer.minMax().repeat(15),
    selectors=[
        "actual_lst",
        "NDVI",
        "NDBI",
        "NDWI",
        "EVI",
        "tree_cover_pct",
        "land_cover_class",
        "population_density",
        "elevation_m",
        "slope_deg",
        "aspect_deg",
        "dist_water_m",
        "air_temp_era5",
        "humidity",
        "wind_speed"
    ]
)

print("June feature statistics:")
print(june_stats.getInfo())

June feature statistics:
{'max': [58.26630938000005, 0.8859749144811858, 0.4294315155725368, 0.23015521064301553, 0.8841928593225509, 16, 80, 200714.83154296875, 309, 45, 356, 22056.835675137085, 35.17293175591362, 42.943623014942176, 3.2017777095513678], 'min': [35.09555180000001, -0.21296645307199397, -0.44001139925904814, -0.8033547466095646, -0.1273612551282629, 0, 10, 0, 193, 0, 0, 0, 34.34935862223307, 39.84219554180398, 3.0236505483043947]}


In [ ]:
# Create the larger June sample for final export preparation

june_samples_large = feature_image_june.sample(
    region=delhi,
    scale=30,
    numPixels=55000,
    seed=42,
    geometries=True
)

print(
    "June large sample created! Number of samples:",
    june_samples_large.size().getInfo()
)

June large sample created! Number of samples: 54697


In [ ]:
# Add metadata to every June sample

def add_june_metadata(feature):
    coords = feature.geometry().coordinates()

    longitude = ee.Number(coords.get(0))
    latitude = ee.Number(coords.get(1))

    cell_id = (
        ee.String("DELHI_JUNE_2024_")
        .cat(longitude.format("%.6f"))
        .cat("_")
        .cat(latitude.format("%.6f"))
    )

    return feature.set({
        "cell_id": cell_id,
        "latitude": latitude,
        "longitude": longitude,
        "month_timestamp": "2024-06-01"
    })


june_samples_final = june_samples_large.map(add_june_metadata)

print(
    "June metadata added! Samples:",
    june_samples_final.size().getInfo()
)

June metadata added! Samples: 54697


In [ ]:
# Inspect one June sample after adding metadata

june_final_first = june_samples_final.first()

print("First June final sample:")
print(june_final_first.getInfo())

First June final sample:
{'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [77.20434884008932, 28.632719970200665]}, 'id': '0', 'properties': {'EVI': 0.23136963389704518, 'NDBI': -0.0747008731270885, 'NDVI': 0.33288770053475936, 'NDWI': -0.3357449088960343, 'actual_lst': 42.81002294000001, 'air_temp_era5': 34.69166547987197, 'aspect_deg': 217, 'cell_id': 'DELHI_JUNE_2024_77.204349_28.632720', 'dist_water_m': 5220, 'elevation_m': 223, 'humidity': 41.61928659698926, 'land_cover_class': 50, 'latitude': 28.632719970200665, 'longitude': 77.20434884008932, 'month_timestamp': '2024-06-01', 'population_density': 19028.785705566406, 'slope_deg': 3, 'tree_cover_pct': 7, 'wind_speed': 3.1479424553375353}}


In [ ]:
# Define the exact 19-column order to match the verified May CSV

june_export_columns = [
    "cell_id",
    "latitude",
    "longitude",
    "month_timestamp",
    "NDVI",
    "NDBI",
    "NDWI",
    "EVI",
    "tree_cover_pct",
    "land_cover_class",
    "population_density",
    "elevation_m",
    "slope_deg",
    "aspect_deg",
    "dist_water_m",
    "air_temp_era5",
    "humidity",
    "wind_speed",
    "actual_lst"
]

print("June export columns:")
print(june_export_columns)
print("Number of columns:", len(june_export_columns))

June export columns:
['cell_id', 'latitude', 'longitude', 'month_timestamp', 'NDVI', 'NDBI', 'NDWI', 'EVI', 'tree_cover_pct', 'land_cover_class', 'population_density', 'elevation_m', 'slope_deg', 'aspect_deg', 'dist_water_m', 'air_temp_era5', 'humidity', 'wind_speed', 'actual_lst']
Number of columns: 19


In [ ]:
# Export final June 2024 dataset to Google Drive

export_task_june = ee.batch.Export.table.toDrive(
    collection=june_samples_final,
    description="Delhi_Urban_Heat_June_2024_FINAL",
    folder="Urban_Heat_AI_ML",
    fileNamePrefix="Delhi_Urban_Heat_June_2024_FINAL",
    fileFormat="CSV",
    selectors=june_export_columns
)

export_task_june.start()

print("Final June export started!")
print("Task ID:", export_task_june.id)

Final June export started!
Task ID: 3QPRQKRHB24HOJS7R66IL4QP


## JUL 2024 DATASET

In [51]:
# =========================
# JULY 2024 - LANDSAT 8 LST
# =========================

landsat_july = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .filterBounds(delhi)
    .filterDate("2024-07-01", "2024-08-01")
)

print("Landsat 8 images in July 2024:", landsat_july.size().getInfo())

Landsat 8 images in July 2024: 6


In [52]:
# Create July 2024 LST composite

def calculate_lst_july(image):
    thermal = image.select("ST_B10")

    lst_celsius = (
        thermal
        .multiply(0.00341802)
        .add(149.0)
        .subtract(273.15)
        .rename("actual_lst")
    )

    return image.addBands(lst_celsius)


landsat_lst_july = landsat_july.map(calculate_lst_july)

lst_july = (
    landsat_lst_july
    .select("actual_lst")
    .median()
    .clip(delhi)
)

print("July 2024 LST composite created!")

July 2024 LST composite created!


In [53]:
# Check July 2024 LST range

july_lst_stats = lst_july.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=delhi,
    scale=30,
    maxPixels=1e9
)

print(july_lst_stats.getInfo())

{'actual_lst_max': 59.640353420000025, 'actual_lst_min': 2.564546450000023}


In [54]:
# Check how many July LST pixels have suspiciously low temperatures

low_20 = lst_july.lt(20).selfMask().reduceRegion(
    reducer=ee.Reducer.count(),
    geometry=delhi,
    scale=30,
    maxPixels=1e9
)

low_10 = lst_july.lt(10).selfMask().reduceRegion(
    reducer=ee.Reducer.count(),
    geometry=delhi,
    scale=30,
    maxPixels=1e9
)

low_5 = lst_july.lt(5).selfMask().reduceRegion(
    reducer=ee.Reducer.count(),
    geometry=delhi,
    scale=30,
    maxPixels=1e9
)

print("Pixels with LST < 20°C:", low_20.getInfo())
print("Pixels with LST < 10°C:", low_10.getInfo())
print("Pixels with LST < 5°C:", low_5.getInfo())

Pixels with LST < 20°C: {'actual_lst': 47544}
Pixels with LST < 10°C: {'actual_lst': 4076}
Pixels with LST < 5°C: {'actual_lst': 610}


In [55]:
# =========================
# JULY 2024 - CLEAN LANDSAT LST
# =========================

def calculate_lst_july_clean(image):
    qa = image.select("QA_PIXEL")
    radsat = image.select("QA_RADSAT")

    # Mask fill, dilated cloud, cirrus, cloud, cloud shadow and snow
    qa_mask = qa.bitwiseAnd(63).eq(0)

    # Mask radiometrically saturated pixels
    radsat_mask = radsat.eq(0)

    final_mask = qa_mask.And(radsat_mask)

    thermal = image.select("ST_B10")

    lst_celsius = (
        thermal
        .multiply(0.00341802)
        .add(149.0)
        .subtract(273.15)
        .rename("actual_lst")
    )

    return lst_celsius.updateMask(final_mask)


landsat_lst_july_clean = landsat_july.map(calculate_lst_july_clean)

lst_july_clean = (
    landsat_lst_july_clean
    .median()
    .clip(delhi)
)

print("Clean July 2024 LST composite created!")

Clean July 2024 LST composite created!


In [56]:
# Check cleaned July 2024 LST range

july_lst_clean_stats = lst_july_clean.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=delhi,
    scale=30,
    maxPixels=1e9
)

print(july_lst_clean_stats.getInfo())


{'actual_lst_max': 59.640353420000025, 'actual_lst_min': 21.372201500000017}


In [57]:
# =========================
# JULY 2024 - SENTINEL-2
# =========================

sentinel_july = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(delhi)
    .filterDate("2024-07-01", "2024-08-01")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30))
)

print(
    "Sentinel-2 images in July 2024:",
    sentinel_july.size().getInfo()
)

Sentinel-2 images in July 2024: 0


In [58]:
# =========================
# JULY 2024 - SENTINEL-2 DIAGNOSTIC
# =========================

sentinel_july_all = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(delhi)
    .filterDate("2024-07-01", "2024-08-01")
)

print(
    "Total Sentinel-2 images in July:",
    sentinel_july_all.size().getInfo()
)

for threshold in [30, 50, 70, 80, 90, 100]:
    count = sentinel_july_all.filter(
        ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", threshold)
    ).size().getInfo()

    print(
        f"Images with CLOUDY_PIXEL_PERCENTAGE < {threshold}%:",
        count
    )

print("\nJuly image cloud percentages:")

cloud_info = sentinel_july_all.aggregate_array(
    "CLOUDY_PIXEL_PERCENTAGE"
).getInfo()

print(cloud_info)

Total Sentinel-2 images in July: 24
Images with CLOUDY_PIXEL_PERCENTAGE < 30%: 0
Images with CLOUDY_PIXEL_PERCENTAGE < 50%: 0
Images with CLOUDY_PIXEL_PERCENTAGE < 70%: 7
Images with CLOUDY_PIXEL_PERCENTAGE < 80%: 9
Images with CLOUDY_PIXEL_PERCENTAGE < 90%: 11
Images with CLOUDY_PIXEL_PERCENTAGE < 100%: 23

July image cloud percentages:
[99.990922, 99.530673, 100, 99.996585, 99.276567, 98.219305, 68.987828, 99.02572, 99.0924, 96.885675, 97.10862, 67.460704, 57.937354, 69.66213, 58.210611, 61.105943, 58.249187, 98.042643, 75.7043, 99.651903, 82.790005, 99.931371, 85.661262, 72.355592]


In [59]:
# =========================
# JULY 2024 - SENTINEL-2 CLEAN COLLECTION
# =========================

sentinel_july_clean = (
    sentinel_july_all
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80))
)

print(
    "Sentinel-2 images used for July 2024:",
    sentinel_july_clean.size().getInfo()
)

print(
    "July scene cloud percentages:",
    sentinel_july_clean.aggregate_array(
        "CLOUDY_PIXEL_PERCENTAGE"
    ).getInfo()
)

Sentinel-2 images used for July 2024: 9
July scene cloud percentages: [68.987828, 67.460704, 57.937354, 69.66213, 58.210611, 61.105943, 58.249187, 75.7043, 72.355592]


In [60]:
# =========================
# JULY 2024 - SENTINEL-2 COMPOSITE
# =========================

sentinel_clean_july = sentinel_july_clean.map(mask_s2)

sentinel_median_july = (
    sentinel_clean_july
    .median()
    .clip(delhi)
)

print("July 2024 Sentinel-2 composite created!")
print(
    "Bands:",
    sentinel_median_july.bandNames().getInfo()
)

July 2024 Sentinel-2 composite created!
Bands: ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE']


In [61]:
# ==========================================
# JULY 2024 — SPECTRAL INDICES
# ==========================================

# 1. NDVI (Vegetation)
ndvi_july = (
    sentinel_median_july
    .normalizedDifference(["B8", "B4"])
    .rename("NDVI")
)

# 2. NDBI (Built-up)
ndbi_july = (
    sentinel_median_july
    .normalizedDifference(["B11", "B8"])
    .rename("NDBI")
)

# 3. NDWI (Water)
ndwi_july = (
    sentinel_median_july
    .normalizedDifference(["B3", "B8"])
    .rename("NDWI")
)

# 4. EVI (Enhanced Vegetation)
# Convert Sentinel-2 scaled bands to reflectance for EVI math
nir_july = sentinel_median_july.select("B8").divide(10000)
red_july = sentinel_median_july.select("B4").divide(10000)
blue_july = sentinel_median_july.select("B2").divide(10000)

# EVI denominator
evi_denominator_july = (
    nir_july
    .add(red_july.multiply(6))
    .subtract(blue_july.multiply(7.5))
    .add(1)
)

# Raw EVI
evi_raw_july = (
    nir_july.subtract(red_july)
    .multiply(2.5)
    .divide(evi_denominator_july)
)

# Keep stable and physically reasonable EVI values (masking out extreme noise)
evi_july = (
    evi_raw_july
    .updateMask(evi_denominator_july.gt(0.1))
    .updateMask(evi_raw_july.gte(-1))
    .updateMask(evi_raw_july.lte(1))
    .rename("EVI")
)

print("July NDVI, NDBI, NDWI and EVI calculated!")

# Check the min/max values to ensure the monsoon clouds didn't break our indices
july_spectral_stats = ee.Image.cat([
    ndvi_july,
    ndbi_july,
    ndwi_july,
    evi_july
]).reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=delhi.geometry(),
    scale=20,
    maxPixels=1e9
)

print(july_spectral_stats.getInfo())

July NDVI, NDBI, NDWI and EVI calculated!
{'EVI_max': 1, 'EVI_min': -0.9801717089125095, 'NDBI_max': 1, 'NDBI_min': -0.707774798927614, 'NDVI_max': 1, 'NDVI_min': -1, 'NDWI_max': 1, 'NDWI_min': -1}


In [62]:
# ==========================================
# JULY 2024 — ERA5-LAND WEATHER
# ==========================================

# 1. Load ERA5-Land hourly data for July 2024
era5_july = (
    ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")
    .filterBounds(delhi)
    .filterDate("2024-07-01", "2024-08-01")
)

print("ERA5-Land images in July 2024:", era5_july.size().getInfo())

# 2. Air Temperature (convert from Kelvin to Celsius)
air_temp_era5_july = (
    era5_july
    .select("temperature_2m")
    .mean()
    .subtract(273.15)
    .rename("air_temp_era5")
    .clip(delhi)
)

# 3. Wind Speed (calculate magnitude from U and V vectors)
def add_wind_speed_july(image):
    u = image.select("u_component_of_wind_10m")
    v = image.select("v_component_of_wind_10m")

    wind = (
        u.pow(2)
        .add(v.pow(2))
        .sqrt()
        .rename("wind_speed")
    )
    return image.addBands(wind)

era5_wind_july = era5_july.map(add_wind_speed_july)

wind_speed_july = (
    era5_wind_july
    .select("wind_speed")
    .mean()
    .rename("wind_speed")
    .clip(delhi)
)

# 4. Relative Humidity (Magnus formula)
def add_relative_humidity_july(image):
    temp_c = image.select("temperature_2m").subtract(273.15)
    dew_c = image.select("dewpoint_temperature_2m").subtract(273.15)

    es = temp_c.expression(
        "6.112 * exp((17.67 * T) / (T + 243.5))",
        {"T": temp_c}
    )

    e = dew_c.expression(
        "6.112 * exp((17.67 * Td) / (Td + 243.5))",
        {"Td": dew_c}
    )

    rh = (
        e.divide(es)
        .multiply(100)
        .rename("humidity")
    )
    return image.addBands(rh)

era5_humidity_july = era5_july.map(add_relative_humidity_july)

humidity_july = (
    era5_humidity_july
    .select("humidity")
    .mean()
    .rename("humidity")
    .clip(delhi)
)

print("July Air Temp, Wind Speed, and Humidity layers created!")

# Validate weather features
weather_features_july = ee.Image.cat([
    air_temp_era5_july,
    humidity_july,
    wind_speed_july
])

weather_stats_july = weather_features_july.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=delhi.geometry(),
    scale=11100, # Using ~11.1km scale corresponding to ERA5 native resolution
    maxPixels=1e9
)

print(weather_stats_july.getInfo())

ERA5-Land images in July 2024: 744
July Air Temp, Wind Speed, and Humidity layers created!
{'air_temp_era5_max': 30.671578835928358, 'air_temp_era5_min': 30.14065698192968, 'humidity_max': 81.106443806038, 'humidity_min': 77.45931491255827, 'wind_speed_max': 1.9674302312408536, 'wind_speed_min': 1.9351133957937123}


In [63]:
# ==========================================
# JULY 2024 — COMPLETE FEATURE IMAGE & SAMPLING
# ==========================================

feature_image_july = ee.Image.cat([
    lst_july_clean,              # actual_lst
    ndvi_july,                   # NDVI
    ndbi_july,                   # NDBI
    ndwi_july,                   # NDWI
    evi_july,                    # EVI
    tree_cover,                  # tree_cover_pct
    worldcover.rename("land_cover_class"),
    population_density,          # population_density
    elevation,                   # elevation_m
    slope,                       # slope_deg
    aspect,                      # aspect_deg
    dist_water,                  # dist_water_m
    air_temp_era5_july,          # air_temp_era5
    humidity_july,               # humidity
    wind_speed_july              # wind_speed
]).clip(delhi)

print("July feature image created!")
print("Bands:", feature_image_july.bandNames().getInfo())

# Generate the test sample to verify before large export
july_samples_test = feature_image_july.sample(
    region=delhi.geometry(),
    scale=30,
    numPixels=5000,
    seed=42,
    geometries=True,
    tileScale=4
)

print("July test sample created! Number of samples:", july_samples_test.size().getInfo())

July feature image created!
Bands: ['actual_lst', 'NDVI', 'NDBI', 'NDWI', 'EVI', 'tree_cover_pct', 'land_cover_class', 'population_density', 'elevation_m', 'slope_deg', 'aspect_deg', 'dist_water_m', 'air_temp_era5', 'humidity', 'wind_speed']
July test sample created! Number of samples: 2274


In [64]:
# ==========================================
# JULY 2024 — LARGE SAMPLE & EXPORT
# ==========================================

# Request a larger initial sample (~120,000) to yield ~50,000 clean pixels after cloud masking
july_samples_large = feature_image_july.sample(
    region=delhi.geometry(),
    scale=30,
    numPixels=120000,
    seed=42,
    geometries=True,
    tileScale=4
)

print("July large sample created! Valid samples:", july_samples_large.size().getInfo())

# Add tracking metadata (cell_id, latitude, longitude, month_timestamp)
def add_july_metadata(feature):
    coords = feature.geometry().coordinates()
    longitude = ee.Number(coords.get(0))
    latitude = ee.Number(coords.get(1))

    cell_id = (
        ee.String("DELHI_JULY_2024_")
        .cat(longitude.format("%.6f"))
        .cat("_")
        .cat(latitude.format("%.6f"))
    )

    return feature.set({
        "cell_id": cell_id,
        "latitude": latitude,
        "longitude": longitude,
        "month_timestamp": "2024-07-01"
    })

july_samples_final = july_samples_large.map(add_july_metadata)

print("July metadata added! Final export count:", july_samples_final.size().getInfo())

# Define the exact 19-column order matching May and June
july_export_columns = [
    "cell_id", "latitude", "longitude", "month_timestamp",
    "NDVI", "NDBI", "NDWI", "EVI", "tree_cover_pct",
    "land_cover_class", "population_density", "elevation_m",
    "slope_deg", "aspect_deg", "dist_water_m", "air_temp_era5",
    "humidity", "wind_speed", "actual_lst"
]

# Start the Google Drive export task
export_task_july = ee.batch.Export.table.toDrive(
    collection=july_samples_final,
    description="Delhi_Urban_Heat_July_2024_FINAL",
    folder="Urban_Heat_AI_ML",
    fileNamePrefix="Delhi_Urban_Heat_July_2024_FINAL",
    fileFormat="CSV",
    selectors=july_export_columns
)

export_task_july.start()

print("Final July export started!")
print("Task ID:", export_task_july.id)

July large sample created! Valid samples: 55616
July metadata added! Final export count: 55616
Final July export started!
Task ID: CAYXUHDYBDVE3BOICD44VYDX


In [65]:
for t in ee.batch.Task.list():
    if t.id == "CAYXUHDYBDVE3BOICD44VYDX":
        print("Task status:", t.status())

Task status: {'state': 'COMPLETED', 'description': 'Delhi_Urban_Heat_July_2024_FINAL', 'priority': 100, 'creation_timestamp_ms': 1790418293968, 'update_timestamp_ms': 1790418457842, 'start_timestamp_ms': 1790418296697, 'task_type': 'EXPORT_FEATURES', 'destination_uris': ['https://drive.google.com/#folders/1oN0NkSA_yCxm1Lkz816FDezcicJhAB7d'], 'attempt': 1, 'batch_eecu_usage_seconds': 5695.39501953125, 'id': 'CAYXUHDYBDVE3BOICD44VYDX', 'name': 'projects/urban-heat-ai-ml/operations/CAYXUHDYBDVE3BOICD44VYDX'}


In [66]:
# ==========================================
# AUGUST 2024 — LANDSAT 8 LST
# ==========================================

landsat_august = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .filterBounds(delhi)
    .filterDate("2024-08-01", "2024-09-01")
)

print("Landsat 8 images in August 2024:", landsat_august.size().getInfo())

def prepare_landsat_lst_august(image):
    qa = image.select("QA_PIXEL")

    # Mask fill, dilated cloud, cirrus, cloud, cloud shadow, snow
    clear_mask = (
        qa.bitwiseAnd(1 << 0).eq(0)
        .And(qa.bitwiseAnd(1 << 1).eq(0))
        .And(qa.bitwiseAnd(1 << 2).eq(0))
        .And(qa.bitwiseAnd(1 << 3).eq(0))
        .And(qa.bitwiseAnd(1 << 4).eq(0))
        .And(qa.bitwiseAnd(1 << 5).eq(0))
    )

    st_qa = image.select("ST_QA")
    quality_mask = st_qa.multiply(0.01).lte(5)

    final_mask = clear_mask.And(quality_mask)

    lst_celsius = (
        image.select("ST_B10")
        .multiply(0.00341802)
        .add(149.0)
        .subtract(273.15)
        .rename("actual_lst")
    )

    return lst_celsius.updateMask(final_mask)

landsat_lst_august = landsat_august.map(prepare_landsat_lst_august)

lst_august_clean = (
    landsat_lst_august
    .median()
    .clip(delhi)
)

august_lst_stats = lst_august_clean.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=delhi.geometry(),
    scale=30,
    maxPixels=1e9
)

print("Clean August 2024 LST composite created!")
print("LST Stats:", august_lst_stats.getInfo())

Landsat 8 images in August 2024: 5
Clean August 2024 LST composite created!
LST Stats: {'actual_lst_max': 53.839973480000026, 'actual_lst_min': 26.365928720000056}


In [67]:
# ==========================================
# AUGUST 2024 — SENTINEL-2 COMPOSITE
# ==========================================

sentinel_august = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(delhi)
    .filterDate("2024-08-01", "2024-09-01")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80))
)

print("Sentinel-2 images used for August 2024:", sentinel_august.size().getInfo())

sentinel_clean_august = sentinel_august.map(mask_s2)

sentinel_median_august = (
    sentinel_clean_august
    .median()
    .clip(delhi)
)

print("August 2024 Sentinel-2 composite created!")
print("Bands:", sentinel_median_august.bandNames().getInfo())

Sentinel-2 images used for August 2024: 15
August 2024 Sentinel-2 composite created!
Bands: ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE']


In [68]:
# ==========================================
# AUGUST 2024 — SPECTRAL INDICES
# ==========================================

# 1. NDVI (Vegetation)
ndvi_august = (
    sentinel_median_august
    .normalizedDifference(["B8", "B4"])
    .rename("NDVI")
)

# 2. NDBI (Built-up)
ndbi_august = (
    sentinel_median_august
    .normalizedDifference(["B11", "B8"])
    .rename("NDBI")
)

# 3. NDWI (Water)
ndwi_august = (
    sentinel_median_august
    .normalizedDifference(["B3", "B8"])
    .rename("NDWI")
)

# 4. EVI (Enhanced Vegetation)
nir_august = sentinel_median_august.select("B8").divide(10000)
red_august = sentinel_median_august.select("B4").divide(10000)
blue_august = sentinel_median_august.select("B2").divide(10000)

evi_denominator_august = (
    nir_august
    .add(red_august.multiply(6))
    .subtract(blue_august.multiply(7.5))
    .add(1)
)

evi_raw_august = (
    nir_august.subtract(red_august)
    .multiply(2.5)
    .divide(evi_denominator_august)
)

evi_august = (
    evi_raw_august
    .updateMask(evi_denominator_august.gt(0.1))
    .updateMask(evi_raw_august.gte(-1))
    .updateMask(evi_raw_august.lte(1))
    .rename("EVI")
)

print("August NDVI, NDBI, NDWI, and EVI calculated!")

# Validate spectral indices bounds
august_spectral_stats = ee.Image.cat([
    ndvi_august,
    ndbi_august,
    ndwi_august,
    evi_august
]).reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=delhi.geometry(),
    scale=20,
    maxPixels=1e9
)

print("Spectral Stats:", august_spectral_stats.getInfo())

August NDVI, NDBI, NDWI, and EVI calculated!
Spectral Stats: {'EVI_max': 0.9999999999999999, 'EVI_min': -0.9572440087145916, 'NDBI_max': 0.5635460251046025, 'NDBI_min': -0.8634751773049646, 'NDVI_max': 0.9640877854134339, 'NDVI_min': -0.6186046511627907, 'NDWI_max': 0.6701030927835051, 'NDWI_min': -0.8785138588559072}


In [69]:
# ==========================================
# AUGUST 2024 — ERA5-LAND WEATHER
# ==========================================

era5_august = (
    ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")
    .filterBounds(delhi)
    .filterDate("2024-08-01", "2024-09-01")
)

print("ERA5-Land images in August 2024:", era5_august.size().getInfo())

# 1. Air Temperature
air_temp_era5_august = (
    era5_august
    .select("temperature_2m")
    .mean()
    .subtract(273.15)
    .rename("air_temp_era5")
    .clip(delhi)
)

# 2. Wind Speed
def add_wind_speed_august(image):
    u = image.select("u_component_of_wind_10m")
    v = image.select("v_component_of_wind_10m")
    wind = u.pow(2).add(v.pow(2)).sqrt().rename("wind_speed")
    return image.addBands(wind)

era5_wind_august = era5_august.map(add_wind_speed_august)
wind_speed_august = (
    era5_wind_august
    .select("wind_speed")
    .mean()
    .rename("wind_speed")
    .clip(delhi)
)

# 3. Relative Humidity
def add_relative_humidity_august(image):
    temp_c = image.select("temperature_2m").subtract(273.15)
    dew_c = image.select("dewpoint_temperature_2m").subtract(273.15)
    es = temp_c.expression("6.112 * exp((17.67 * T) / (T + 243.5))", {"T": temp_c})
    e = dew_c.expression("6.112 * exp((17.67 * Td) / (Td + 243.5))", {"Td": dew_c})
    rh = e.divide(es).multiply(100).rename("humidity")
    return image.addBands(rh)

era5_humidity_august = era5_august.map(add_relative_humidity_august)
humidity_august = (
    era5_humidity_august
    .select("humidity")
    .mean()
    .rename("humidity")
    .clip(delhi)
)

print("August Air Temp, Wind Speed, and Humidity layers created!")

# Validate weather features
weather_features_august = ee.Image.cat([
    air_temp_era5_august,
    humidity_august,
    wind_speed_august
])

august_weather_stats = weather_features_august.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=delhi.geometry(),
    scale=11100,
    maxPixels=1e9
)

print("Weather Stats:", august_weather_stats.getInfo())

ERA5-Land images in August 2024: 744
August Air Temp, Wind Speed, and Humidity layers created!
Weather Stats: {'air_temp_era5_max': 28.361122980425478, 'air_temp_era5_min': 28.104596812750742, 'humidity_max': 84.69372132573184, 'humidity_min': 83.80575406395131, 'wind_speed_max': 2.4004202749624675, 'wind_speed_min': 2.259139156108706}


In [70]:
# ==========================================
# AUGUST 2024 — COMBINATION & TEST SAMPLE
# ==========================================

feature_image_august = ee.Image.cat([
    lst_august_clean,
    ndvi_august,
    ndbi_august,
    ndwi_august,
    evi_august,
    tree_cover,
    worldcover.rename("land_cover_class"),
    population_density,
    elevation,
    slope,
    aspect,
    dist_water,
    air_temp_era5_august,
    humidity_august,
    wind_speed_august
]).clip(delhi)

print("August complete feature image created!")
print("Bands:", feature_image_august.bandNames().getInfo())

august_samples_test = feature_image_august.sample(
    region=delhi.geometry(),
    scale=30,
    numPixels=5000,
    seed=42,
    geometries=True,
    tileScale=4
)

print("August test sample created! Number of valid samples:", august_samples_test.size().getInfo())

August complete feature image created!
Bands: ['actual_lst', 'NDVI', 'NDBI', 'NDWI', 'EVI', 'tree_cover_pct', 'land_cover_class', 'population_density', 'elevation_m', 'slope_deg', 'aspect_deg', 'dist_water_m', 'air_temp_era5', 'humidity', 'wind_speed']
August test sample created! Number of valid samples: 2357


In [71]:
# ==========================================
# AUGUST 2024 — LARGE SAMPLE & EXPORT
# ==========================================

# Request a larger initial sample (~120,000) to yield ~50,000 clean pixels
august_samples_large = feature_image_august.sample(
    region=delhi.geometry(),
    scale=30,
    numPixels=120000,
    seed=42,
    geometries=True,
    tileScale=4
)

print("August large sample created! Valid samples:", august_samples_large.size().getInfo())

# Add tracking metadata
def add_august_metadata(feature):
    coords = feature.geometry().coordinates()
    longitude = ee.Number(coords.get(0))
    latitude = ee.Number(coords.get(1))

    cell_id = (
        ee.String("DELHI_AUG_2024_")
        .cat(longitude.format("%.6f"))
        .cat("_")
        .cat(latitude.format("%.6f"))
    )

    return feature.set({
        "cell_id": cell_id,
        "latitude": latitude,
        "longitude": longitude,
        "month_timestamp": "2024-08-01"
    })

august_samples_final = august_samples_large.map(add_august_metadata)

print("August metadata added! Final export count:", august_samples_final.size().getInfo())

# Define the exact 19-column order matching previous months
august_export_columns = [
    "cell_id", "latitude", "longitude", "month_timestamp",
    "NDVI", "NDBI", "NDWI", "EVI", "tree_cover_pct",
    "land_cover_class", "population_density", "elevation_m",
    "slope_deg", "aspect_deg", "dist_water_m", "air_temp_era5",
    "humidity", "wind_speed", "actual_lst"
]

# Start the Google Drive export task
export_task_august = ee.batch.Export.table.toDrive(
    collection=august_samples_final,
    description="Delhi_Urban_Heat_August_2024_FINAL",
    folder="Urban_Heat_AI_ML",
    fileNamePrefix="Delhi_Urban_Heat_August_2024_FINAL",
    fileFormat="CSV",
    selectors=august_export_columns
)

export_task_august.start()

print("Final August export started!")
print("Task ID:", export_task_august.id)

August large sample created! Valid samples: 55615
August metadata added! Final export count: 55615
Final August export started!
Task ID: DZ3JHIHNBY3JARHV76J54GGJ


In [72]:
for t in ee.batch.Task.list():
    if t.id == "DZ3JHIHNBY3JARHV76J54GGJ":
        print("Task status:", t.status())

Task status: {'state': 'COMPLETED', 'description': 'Delhi_Urban_Heat_August_2024_FINAL', 'priority': 100, 'creation_timestamp_ms': 1790419768432, 'update_timestamp_ms': 1790419786879, 'start_timestamp_ms': 1790419773672, 'task_type': 'EXPORT_FEATURES', 'destination_uris': ['https://drive.google.com/#folders/1oN0NkSA_yCxm1Lkz816FDezcicJhAB7d'], 'attempt': 1, 'batch_eecu_usage_seconds': 3.9181461334228516, 'id': 'DZ3JHIHNBY3JARHV76J54GGJ', 'name': 'projects/urban-heat-ai-ml/operations/DZ3JHIHNBY3JARHV76J54GGJ'}


In [73]:
# ==========================================
# SEPTEMBER 2024 — LANDSAT 8 LST
# ==========================================

landsat_september = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .filterBounds(delhi)
    .filterDate("2024-09-01", "2024-10-01")
)

print("Landsat 8 images in September 2024:", landsat_september.size().getInfo())

def prepare_landsat_lst_september(image):
    qa = image.select("QA_PIXEL")

    # Mask fill, dilated cloud, cirrus, cloud, cloud shadow, snow
    clear_mask = (
        qa.bitwiseAnd(1 << 0).eq(0)
        .And(qa.bitwiseAnd(1 << 1).eq(0))
        .And(qa.bitwiseAnd(1 << 2).eq(0))
        .And(qa.bitwiseAnd(1 << 3).eq(0))
        .And(qa.bitwiseAnd(1 << 4).eq(0))
        .And(qa.bitwiseAnd(1 << 5).eq(0))
    )

    st_qa = image.select("ST_QA")
    quality_mask = st_qa.multiply(0.01).lte(5)

    final_mask = clear_mask.And(quality_mask)

    lst_celsius = (
        image.select("ST_B10")
        .multiply(0.00341802)
        .add(149.0)
        .subtract(273.15)
        .rename("actual_lst")
    )

    return lst_celsius.updateMask(final_mask)

landsat_lst_september = landsat_september.map(prepare_landsat_lst_september)

lst_september_clean = (
        landsat_lst_september
        .median()
        .clip(delhi)
)

september_lst_stats = lst_september_clean.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=delhi.geometry(),
    scale=30,
    maxPixels=1e9
)

print("Clean September 2024 LST composite created!")
print("LST Stats:", september_lst_stats.getInfo())

Landsat 8 images in September 2024: 3
Clean September 2024 LST composite created!
LST Stats: {'actual_lst_max': 45.60938132000001, 'actual_lst_min': 24.909852200000046}


In [74]:
# ==========================================
# SEPTEMBER 2024 — SENTINEL-2 COMPOSITE
# ==========================================

sentinel_september = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(delhi)
    .filterDate("2024-09-01", "2024-10-01")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80))
)

print("Sentinel-2 images used for September 2024:", sentinel_september.size().getInfo())

sentinel_clean_september = sentinel_september.map(mask_s2)

sentinel_median_september = (
    sentinel_clean_september
    .median()
    .clip(delhi)
)

print("September 2024 Sentinel-2 composite created!")
print("Bands:", sentinel_median_september.bandNames().getInfo())

Sentinel-2 images used for September 2024: 22
September 2024 Sentinel-2 composite created!
Bands: ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE']


In [75]:
# ==========================================
# SEPTEMBER 2024 — SPECTRAL INDICES
# ==========================================

# 1. NDVI (Vegetation)
ndvi_september = (
    sentinel_median_september
    .normalizedDifference(["B8", "B4"])
    .rename("NDVI")
)

# 2. NDBI (Built-up)
ndbi_september = (
    sentinel_median_september
    .normalizedDifference(["B11", "B8"])
    .rename("NDBI")
)

# 3. NDWI (Water)
ndwi_september = (
    sentinel_median_september
    .normalizedDifference(["B3", "B8"])
    .rename("NDWI")
)

# 4. EVI (Enhanced Vegetation)
nir_september = sentinel_median_september.select("B8").divide(10000)
red_september = sentinel_median_september.select("B4").divide(10000)
blue_september = sentinel_median_september.select("B2").divide(10000)

evi_denominator_september = (
    nir_september
    .add(red_september.multiply(6))
    .subtract(blue_september.multiply(7.5))
    .add(1)
)

evi_raw_september = (
    nir_september.subtract(red_september)
    .multiply(2.5)
    .divide(evi_denominator_september)
)

evi_september = (
    evi_raw_september
    .updateMask(evi_denominator_september.gt(0.1))
    .updateMask(evi_raw_september.gte(-1))
    .updateMask(evi_raw_september.lte(1))
    .rename("EVI")
)

print("September NDVI, NDBI, NDWI, and EVI calculated!")

# Validate spectral indices bounds
september_spectral_stats = ee.Image.cat([
    ndvi_september,
    ndbi_september,
    ndwi_september,
    evi_september
]).reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=delhi.geometry(),
    scale=20,
    maxPixels=1e9
)

print("Spectral Stats:", september_spectral_stats.getInfo())

September NDVI, NDBI, NDWI, and EVI calculated!
Spectral Stats: {'EVI_max': 0.9998777281897658, 'EVI_min': -0.8380102040816335, 'NDBI_max': 0.6517353900919608, 'NDBI_min': -0.8373071528751753, 'NDVI_max': 0.9289642529789184, 'NDVI_min': -0.5180277349768875, 'NDWI_max': 0.6274416388756551, 'NDWI_min': -0.8302170642596175}


In [76]:
# ==========================================
# SEPTEMBER 2024 — ERA5-LAND WEATHER
# ==========================================

era5_september = (
    ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")
    .filterBounds(delhi)
    .filterDate("2024-09-01", "2024-10-01")
)

print("ERA5-Land images in September 2024:", era5_september.size().getInfo())

# 1. Air Temperature
air_temp_era5_september = (
    era5_september
    .select("temperature_2m")
    .mean()
    .subtract(273.15)
    .rename("air_temp_era5")
    .clip(delhi)
)

# 2. Wind Speed
def add_wind_speed_september(image):
    u = image.select("u_component_of_wind_10m")
    v = image.select("v_component_of_wind_10m")
    wind = u.pow(2).add(v.pow(2)).sqrt().rename("wind_speed")
    return image.addBands(wind)

era5_wind_september = era5_september.map(add_wind_speed_september)
wind_speed_september = (
    era5_wind_september
    .select("wind_speed")
    .mean()
    .rename("wind_speed")
    .clip(delhi)
)

# 3. Relative Humidity
def add_relative_humidity_september(image):
    temp_c = image.select("temperature_2m").subtract(273.15)
    dew_c = image.select("dewpoint_temperature_2m").subtract(273.15)
    es = temp_c.expression("6.112 * exp((17.67 * T) / (T + 243.5))", {"T": temp_c})
    e = dew_c.expression("6.112 * exp((17.67 * Td) / (Td + 243.5))", {"Td": dew_c})
    rh = e.divide(es).multiply(100).rename("humidity")
    return image.addBands(rh)

era5_humidity_september = era5_september.map(add_relative_humidity_september)
humidity_september = (
    era5_humidity_september
    .select("humidity")
    .mean()
    .rename("humidity")
    .clip(delhi)
)

print("September Air Temp, Wind Speed, and Humidity layers created!")

# Validate weather features
weather_features_september = ee.Image.cat([
    air_temp_era5_september,
    humidity_september,
    wind_speed_september
])

september_weather_stats = weather_features_september.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=delhi.geometry(),
    scale=11100,
    maxPixels=1e9
)

print("Weather Stats:", september_weather_stats.getInfo())

ERA5-Land images in September 2024: 720
September Air Temp, Wind Speed, and Humidity layers created!
Weather Stats: {'air_temp_era5_max': 27.613394271002903, 'air_temp_era5_min': 27.33681006961399, 'humidity_max': 83.64826026860565, 'humidity_min': 82.29665500165326, 'wind_speed_max': 2.0430461543986946, 'wind_speed_min': 1.9620238984299117}


In [77]:
# ==========================================
# SEPTEMBER 2024 — COMBINATION, SAMPLE & EXPORT
# ==========================================

feature_image_september = ee.Image.cat([
    lst_september_clean,
    ndvi_september,
    ndbi_september,
    ndwi_september,
    evi_september,
    tree_cover,
    worldcover.rename("land_cover_class"),
    population_density,
    elevation,
    slope,
    aspect,
    dist_water,
    air_temp_era5_september,
    humidity_september,
    wind_speed_september
]).clip(delhi)

print("September complete feature image created!")

# Request the large initial sample
september_samples_large = feature_image_september.sample(
    region=delhi.geometry(),
    scale=30,
    numPixels=120000,
    seed=42,
    geometries=True,
    tileScale=4
)

print("September large sample created! Valid samples:", september_samples_large.size().getInfo())

# Add tracking metadata
def add_september_metadata(feature):
    coords = feature.geometry().coordinates()
    longitude = ee.Number(coords.get(0))
    latitude = ee.Number(coords.get(1))

    cell_id = (
        ee.String("DELHI_SEP_2024_")
        .cat(longitude.format("%.6f"))
        .cat("_")
        .cat(latitude.format("%.6f"))
    )

    return feature.set({
        "cell_id": cell_id,
        "latitude": latitude,
        "longitude": longitude,
        "month_timestamp": "2024-09-01"
    })

september_samples_final = september_samples_large.map(add_september_metadata)
print("September metadata added! Final export count:", september_samples_final.size().getInfo())

# Enforce the 19-column schema
september_export_columns = [
    "cell_id", "latitude", "longitude", "month_timestamp",
    "NDVI", "NDBI", "NDWI", "EVI", "tree_cover_pct",
    "land_cover_class", "population_density", "elevation_m",
    "slope_deg", "aspect_deg", "dist_water_m", "air_temp_era5",
    "humidity", "wind_speed", "actual_lst"
]

# Start the Google Drive export task
export_task_september = ee.batch.Export.table.toDrive(
    collection=september_samples_final,
    description="Delhi_Urban_Heat_September_2024_FINAL",
    folder="Urban_Heat_AI_ML",
    fileNamePrefix="Delhi_Urban_Heat_September_2024_FINAL",
    fileFormat="CSV",
    selectors=september_export_columns
)

export_task_september.start()
print("Final September export started! Task ID:", export_task_september.id)

September complete feature image created!
September large sample created! Valid samples: 116925
September metadata added! Final export count: 116925
Final September export started! Task ID: 22JLSEIOGOPJVLD7M6VZKLNN


In [79]:
for t in ee.batch.Task.list():
    if t.id == "22JLSEIOGOPJVLD7M6VZKLNN":
        print("Task status:", t.status())

Task status: {'state': 'COMPLETED', 'description': 'Delhi_Urban_Heat_September_2024_FINAL', 'priority': 100, 'creation_timestamp_ms': 1790420934308, 'update_timestamp_ms': 1790421109193, 'start_timestamp_ms': 1790420938451, 'task_type': 'EXPORT_FEATURES', 'destination_uris': ['https://drive.google.com/#folders/1oN0NkSA_yCxm1Lkz816FDezcicJhAB7d'], 'attempt': 1, 'batch_eecu_usage_seconds': 5381.59033203125, 'id': '22JLSEIOGOPJVLD7M6VZKLNN', 'name': 'projects/urban-heat-ai-ml/operations/22JLSEIOGOPJVLD7M6VZKLNN'}


In [81]:
import pandas as pd
import glob
import os
from google.colab import drive

# 1. Actively mount Google Drive (Required)
drive.mount('/content/drive')

# 2. Define the folder path
drive_folder = "/content/drive/MyDrive/Urban_Heat_AI_ML/"

# 3. Locate all 5 monthly CSV files
all_files = glob.glob(os.path.join(drive_folder, "Delhi_Urban_Heat_*_2024_FINAL.csv"))
print(f"Found {len(all_files)} files to combine.")

# 4. Safety Check & Concatenation
if len(all_files) == 0:
    print("ERROR: No files found. Please verify the folder path in your Google Drive.")
else:
    df_list = [pd.read_csv(file) for file in all_files]
    master_df = pd.concat(df_list, ignore_index=True)

    print("\n--- MASTER DATASET INTEGRITY ---")
    print("Total records:", master_df.shape[0])
    print("Total columns:", master_df.shape[1])
    print("Missing values:", master_df.isnull().sum().sum())

    unique_cells = master_df['cell_id'].nunique()
    print(f"Unique grid cells (cell_id): {unique_cells}")

    output_parquet = os.path.join(drive_folder, "Delhi_Urban_Heat_Master_2024.parquet")
    output_csv = os.path.join(drive_folder, "Delhi_Urban_Heat_Master_2024.csv")

    master_df.to_parquet(output_parquet, index=False)
    master_df.to_csv(output_csv, index=False)

    print(f"\nMaster dataset successfully saved to: {output_parquet}")

Mounted at /content/drive
Found 5 files to combine.

--- MASTER DATASET INTEGRITY ---
Total records: 337590
Total columns: 19
Missing values: 0
Unique grid cells (cell_id): 337590

Master dataset successfully saved to: /content/drive/MyDrive/Urban_Heat_AI_ML/Delhi_Urban_Heat_Master_2024.parquet


In [82]:
import pandas as pd
import os

drive_folder = "/content/drive/MyDrive/Urban_Heat_AI_ML/"
master_csv = os.path.join(drive_folder, "Delhi_Urban_Heat_Master_2024.csv")

try:
    # Load the master dataset
    print("Loading Master Dataset...")
    df = pd.read_csv(master_csv)

    # Core Integrity Checks
    print("\n--- MASTER DATASET VALIDATION ---")
    print(f"Total Rows: {df.shape[0]} (Target: ~337,590)")
    print(f"Total Columns: {df.shape[1]} (Target: 19)")
    print(f"Total Missing Values: {df.isnull().sum().sum()}")

    # Check if all 5 months are present
    print("\nMonths included in this master file:")
    print(df['month_timestamp'].value_counts().sort_index())

except Exception as e:
    print(f"Error loading file: {e}")

Loading Master Dataset...

--- MASTER DATASET VALIDATION ---
Total Rows: 337590 (Target: ~337,590)
Total Columns: 19 (Target: 19)
Total Missing Values: 0

Months included in this master file:
month_timestamp
2024-05-01     54737
2024-06-01     54697
2024-07-01     55616
2024-08-01     55615
2024-09-01    116925
Name: count, dtype: int64


In [83]:
import subprocess
import sys

# Ensure pyarrow is installed for Parquet support
try:
    import pyarrow
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyarrow"])

import pandas as pd
import os

drive_folder = "/content/drive/MyDrive/Urban_Heat_AI_ML/"
master_parquet = os.path.join(drive_folder, "Delhi_Urban_Heat_Master_2024.parquet")

try:
    print("Loading Parquet Master Dataset...")
    df_parquet = pd.read_parquet(master_parquet)

    print("\n--- PARQUET MASTER DATASET VALIDATION ---")
    print(f"Total Rows: {df_parquet.shape[0]} (Target: 337,590)")
    print(f"Total Columns: {df_parquet.shape[1]} (Target: 19)")
    print(f"Total Missing Values: {df_parquet.isnull().sum().sum()}")

    print("\nMonths included in Parquet file:")
    print(df_parquet['month_timestamp'].value_counts().sort_index())

except Exception as e:
    print(f"Error loading Parquet file: {e}")

Loading Parquet Master Dataset...

--- PARQUET MASTER DATASET VALIDATION ---
Total Rows: 337590 (Target: 337,590)
Total Columns: 19 (Target: 19)
Total Missing Values: 0

Months included in Parquet file:
month_timestamp
2024-05-01     54737
2024-06-01     54697
2024-07-01     55616
2024-08-01     55615
2024-09-01    116925
Name: count, dtype: int64
